In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, udf
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator, RegressionEvaluator

spark = SparkSession.builder \
    .appName("olist_classification") \
    .getOrCreate()

hdfs_path = "hdfs://localhost:9000/bigdata/"

orders_df = spark.read.csv(hdfs_path + "olist_orders_dataset.csv", header=True, inferSchema=True)
items_df = spark.read.csv(hdfs_path + "olist_order_items_dataset.csv", header=True, inferSchema=True)
products_df = spark.read.csv(hdfs_path + "olist_products_dataset.csv", header=True, inferSchema=True)
customers_df = spark.read.csv(hdfs_path + "olist_customers_dataset.csv", header=True, inferSchema=True)

In [2]:
orders_clean = orders_df.filter(
    (col("order_status") == "delivered") & 
    col("order_delivered_customer_date").isNotNull() & 
    col("order_estimated_delivery_date").isNotNull()
)

orders_with_label = orders_clean.withColumn(
    "label", 
    when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1).otherwise(0)
)

df_joined = orders_with_label.join(items_df, "order_id", "inner") \
                             .join(products_df, "product_id", "inner") \
                             .join(customers_df, "customer_id", "inner")

raw_data = df_joined.select(
    "customer_state", 
    "product_category_name", 
    col("price").cast("double"), 
    col("freight_value").cast("double"), 
    "label"
).dropna()

train_data, test_data = raw_data.randomSplit([0.8, 0.2], seed=42)
train_data.cache()

DataFrame[customer_state: string, product_category_name: string, price: double, freight_value: double, label: int]

In [3]:
state_indexer = StringIndexer(inputCol="customer_state", outputCol="state_indexed", handleInvalid="keep")
category_indexer = StringIndexer(inputCol="product_category_name", outputCol="category_indexed", handleInvalid="keep")

state_encoder = OneHotEncoder(inputCol="state_indexed", outputCol="state_encoded")
category_encoder = OneHotEncoder(inputCol="category_indexed", outputCol="category_encoded")

numerical_cols = ["price", "freight_value"]
categorical_encoded_cols = ["state_encoded", "category_encoded"]
assembler = VectorAssembler(inputCols=numerical_cols + categorical_encoded_cols, outputCol="features")

preprocessing_stages = [state_indexer, category_indexer, state_encoder, category_encoder, assembler]

In [ ]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
rf_pipeline = Pipeline(stages=preprocessing_stages + [rf])
rf_model = rf_pipeline.fit(train_data)

rf_predictions = rf_model.transform(test_data)

get_prob = udf(lambda v: float(v[1]), DoubleType())
rf_predictions_eval = rf_predictions.withColumn("prob_class_1", get_prob(col("probability")))

acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
roc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability", metricName="areaUnderROC")
rmse_eval = RegressionEvaluator(labelCol="label", predictionCol="prob_class_1", metricName="rmse")

print("--- KẾT QUẢ RANDOM FOREST ---")
print(f"Accuracy:  {acc_eval.evaluate(rf_predictions):.4f}")
print(f"F1-Score:  {f1_eval.evaluate(rf_predictions):.4f}")
print(f"ROC-AUC:   {roc_eval.evaluate(rf_predictions):.4f}")
print(f"RMSE:      {rmse_eval.evaluate(rf_predictions_eval):.4f}")

--- KẾT QUẢ RANDOM FOREST ---
Accuracy:  0.9192
F1-Score:  0.8804
ROC-AUC:   0.6191


Py4JJavaError: An error occurred while calling o1555.evaluate.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 74.0 failed 1 times, most recent failure: Lost task 1.0 in stage 74.0 (TID 199) (host.docker.internal executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:281)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.sql.execution.python.BatchEvalPythonEvaluatorFactory.evaluate(BatchEvalPythonExec.scala:95)
	at org.apache.spark.sql.execution.python.EvalPythonEvaluatorFactory$EvalPythonPartitionEvaluator.eval(EvalPythonEvaluatorFactory.scala:113)
	at org.apache.spark.sql.execution.python.EvalPythonExec.$

In [ ]:
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=30, maxDepth=5, seed=42)
gbt_pipeline = Pipeline(stages=preprocessing_stages + [gbt])
gbt_model = gbt_pipeline.fit(train_data)

gbt_predictions = gbt_model.transform(test_data)
gbt_predictions_eval = gbt_predictions.withColumn("prob_class_1", get_prob(col("probability")))

print("--- KẾT QUẢ GRADIENT-BOOTED TREES ---")
print(f"Accuracy:  {acc_eval.evaluate(gbt_predictions):.4f}")
print(f"F1-Score:  {f1_eval.evaluate(gbt_predictions):.4f}")
print(f"ROC-AUC:   {roc_eval.evaluate(gbt_predictions):.4f}")
print(f"RMSE:      {rmse_eval.evaluate(gbt_predictions_eval):.4f}")